In [2]:
!git clone https://github.com/apple/ml-depth-pro.git

fatal: destination path 'ml-depth-pro' already exists and is not an empty directory.


In [3]:
!pip install ml-depth-pro/.
!pip install huggingface-hub
!huggingface-cli download --local-dir checkpoints apple/DepthPro
!pip install torch torchvision

Processing ./ml-depth-pro
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for depth_pro: filename=depth_pro-0.1-py3-none-any.whl size=27528 sha256=89f7dab11b15a0c7db90478959e32c2ab21b62247db4c75734dfb36743ed8416
  Stored in directory: /root/.cache/pip/wheels/38/99/fb/01469566e17668b0abb24c6fdec3a63988f218606d53936ebc
Successfully built depth_pro
  Attempting uninstall: depth_pro
    Found existing installation: depth_pro 0.1
    Uninstalling depth_pro-0.1:
      Successfully uninstalled depth_pro-0.1
Fetching 4 files: 100%|█████████████████████████| 4/4 [00:00<00:00, 2001.58it/s]
/kaggle/working/checkpoints


In [4]:
!huggingface-cli download --local-dir checkpoints apple/DepthPro

Fetching 4 files: 100%|█████████████████████████| 4/4 [00:00<00:00, 1718.98it/s]
/kaggle/working/checkpoints


In [5]:
import torch
import depth_pro
import os
import numpy as np
from PIL import Image
from tqdm import tqdm

# Verify CUDA availability and print device information
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("CUDA is available. Using device:", torch.cuda.get_device_name(0))
    # torch.backends.cudnn.benchmark = True  # Enable cuDNN auto-tuner for performance
else:
    device = torch.device("cpu")
    print("CUDA is not available. Running on CPU.")

# Load model and transforms
model, transform = depth_pro.create_model_and_transforms()
model.eval().to(device)  # Move model to the device

# Dataset paths
image_dir = "/kaggle/input/provided-images"
output_dir = "/kaggle/working/depth-maps"
os.makedirs(output_dir, exist_ok=True)

# Process images with progress bar
images = os.listdir(image_dir)
for img_name in tqdm(images, desc="Processing images"):
    img_path = os.path.join(image_dir, img_name)
    # Load image and retrieve intrinsic parameter f_px
    image, _, f_px = depth_pro.load_rgb(img_path)
    # Apply transform, add batch dimension, and move to device
    image_tensor = transform(image).unsqueeze(0).to(device)
    # Run inference with f_px provided
    prediction = model.infer(image_tensor, f_px=f_px)
    depth_map = prediction["depth"]

    # Convert tensor to numpy, normalize, and then to a PIL image
    depth_map = depth_map.squeeze().cpu().numpy()
    depth_map = (depth_map - depth_map.min()) / (depth_map.max() - depth_map.min() + 1e-8)
    depth_map_img = Image.fromarray((depth_map * 255).astype("uint8"))
    
    output_path = os.path.join(output_dir, f"depth_{img_name}")
    depth_map_img.save(output_path)


CUDA is available. Using device: Tesla P100-PCIE-16GB


Processing images: 100%|██████████| 458/458 [21:26<00:00,  2.81s/it]


In [7]:
!zip -r maps.zip /kaggle/working/depth-maps

updating: kaggle/working/depth-maps/ (stored 0%)
updating: kaggle/working/depth-maps/depth_79316008.jpg (deflated 8%)
updating: kaggle/working/depth-maps/depth_93349216.jpg (deflated 17%)
updating: kaggle/working/depth-maps/depth_110449070.jpg (deflated 17%)
updating: kaggle/working/depth-maps/depth_74849368.jpg (deflated 9%)
updating: kaggle/working/depth-maps/depth_107282469.jpg (deflated 7%)
updating: kaggle/working/depth-maps/depth_102249153.jpg (deflated 14%)
updating: kaggle/working/depth-maps/depth_65549443.jpg (deflated 21%)
updating: kaggle/working/depth-maps/depth_93549223.jpg (deflated 8%)
updating: kaggle/working/depth-maps/depth_64716106.jpg (deflated 23%)
updating: kaggle/working/depth-maps/depth_82982676.jpg (deflated 12%)
updating: kaggle/working/depth-maps/depth_90582579.jpg (deflated 8%)
updating: kaggle/working/depth-maps/depth_78949348.jpg (deflated 8%)
updating: kaggle/working/depth-maps/depth_87515931.jpg (deflated 9%)
updating: kaggle/working/depth-maps/depth_604

In [9]:
from IPython.display import FileLink
FileLink(r'maps.zip')

/kaggle/working/maps.zip